In [ ]:
## MANUAL BATCHING APPROACH

In [ ]:
SET SESSION wait_timeout = 3600;  -- 1 hour
SET SESSION interactive_timeout = 3600;
SET SESSION net_read_timeout = 600;  -- 10 minutes
SET SESSION net_write_timeout = 600;
SET SESSION innodb_lock_wait_timeout = 600; 

In [ ]:
## (1) raw layer: check existing dates
SELECT DISTINCT date, COUNT(*) FROM raw_atspm GROUP BY date;

## (2) derived layer: check existing dates
SELECT DISTINCT date FROM derived_atspm;
SELECT DISTINCT date, COUNT(*) FROM derived_atspm GROUP BY date;
## (2a) add next "batch" (~8 min for 4 days)
CALL process_derived('2024-04-05', '2024-04-05');
## (2b) check added dates (expected dates, consistent data volume)
SELECT DISTINCT date FROM derived_atspm;
SELECT DISTINCT date, COUNT(*) FROM derived_atspm GROUP BY date;

## (3) time layer: check existing dates
SELECT DISTINCT date, COUNT(*) FROM aggr_5min GROUP BY date;
SELECT DISTINCT date, COUNT(*) FROM aggr_15min GROUP BY date;
SELECT DISTINCT date, COUNT(*) FROM aggr_60min GROUP BY date;
## (3a) add next "batch" (~3.5 min for 4 days)
CALL process_time_aggr('2024-04-05', '2024-04-07');
## (3b) check added dates
SELECT DISTINCT date, COUNT(*) FROM aggr_5min GROUP BY date;
SELECT DISTINCT date, COUNT(*) FROM aggr_15min GROUP BY date;
SELECT DISTINCT date, COUNT(*) FROM aggr_60min GROUP BY date;

## (4) lane metrics layer: check existing dates
SELECT DISTINCT date, COUNT(*) FROM lane_metrics GROUP BY date;
## (3a) add next "batch" (~4.5 min for 4 days)
CALL process_lane_metrics('2024-04-05', '2024-04-07');
## (3b) check added dates
SELECT DISTINCT date, COUNT(*) FROM lane_metrics GROUP BY date;

In [ ]:
## IF TIMEOUT
SET SESSION innodb_lock_wait_timeout = 180; 
DELETE FROM derived_atspm WHERE date = '2024-04-07';
COMMIT;

-- For temporary adjustment (until next restart):
SET GLOBAL innodb_buffer_pool_size = 4*1024*1024*1024;  -- 4GB
SET GLOBAL max_connections = 50;  -- Reduce if you don't need many connections


SET SESSION innodb_lock_wait_timeout = 600; 
DELETE FROM derived_atspm WHERE date = '2024-04-06';
COMMIT;

SET SESSION innodb_lock_wait_timeout = 120; -- Just for your current connection

SHOW VARIABLES LIKE 'innodb_buffer_pool_size';
SET GLOBAL innodb_buffer_pool_size=4*1024*1024*1024; -- 4GB example

SET GLOBAL tmp_table_size = 256*1024*1024;
SET GLOBAL max_heap_table_size = 256*1024*1024;
SET GLOBAL sort_buffer_size = 4*1024*1024;

In [ ]:
## other approach

TRUNCATE TABLE derived_atspm;

-- Before load:
ALTER TABLE derived_atspm DROP INDEX idx_15min, 
                         DROP INDEX idx_60min_peak,
                         DROP INDEX idx_lane_grouping,
                         DROP INDEX idx_time_5min,
                         DROP INDEX idx_time_15min,
                         DROP INDEX idx_time_60min,
                         DROP INDEX idx_peak_period;

-- Load in weekly batches
CALL process_derived('2024-04-01', '2024-04-07', 1);
CALL process_derived('2024-04-08', '2024-04-14', 1);
CALL process_derived('2024-04-15', '2024-04-22', 1);
CALL process_derived('2024-04-23', '2024-04-30', 1);
CALL process_derived('2024-05-15', '2024-05-15', 1);
CALL process_derived('2024-06-01', '2024-06-07', 1);
CALL process_derived('2024-06-08', '2024-06-14', 1);
CALL process_derived('2024-06-15', '2024-06-22', 1);
CALL process_derived('2024-06-23', '2024-06-30', 1);
-- ... etc ...

ANALYZE TABLE derived_atspm;

-- After load:
SHOW INDEXES FROM derived_atspm;
-- For aggr_5min query (covers filtering and grouping)
ALTER TABLE derived_atspm ADD INDEX idx_5min_export (date, signalid, time_5min, direction, lane_grouping, volume);
ALTER TABLE derived_atspm ADD INDEX idx_15min_export (date, signalid, time_15min, direction, lane_grouping, volume);
ALTER TABLE derived_atspm ADD INDEX idx_60min_export (date, signalid, time_60min, direction, lane_grouping, volume);


ANALYZE TABLE derived_atspm;

TRUNCATE TABLE aggr_5min;
TRUNCATE TABLE aggr_15min;
TRUNCATE TABLE aggr_60min;

ALTER TABLE aggr_5min DROP INDEX idx_export_order;
ALTER TABLE aggr_15min DROP INDEX idx_export_order;
ALTER TABLE aggr_60min DROP INDEX idx_export_order;

CALL process_time_aggr('2024-04-01', '2024-04-07', 1);
CALL process_time_aggr('2024-04-08', '2024-04-14', 1);
CALL process_time_aggr('2024-04-15', '2024-04-22', 1);
CALL process_time_aggr('2024-04-23', '2024-04-30', 1);
CALL process_time_aggr('2024-05-15', '2024-05-15', 1);
CALL process_time_aggr('2024-06-01', '2024-06-07', 1);
CALL process_time_aggr('2024-06-08', '2024-06-14', 1);
CALL process_time_aggr('2024-06-15', '2024-06-22', 1);
CALL process_time_aggr('2024-06-23', '2024-06-30', 1);

ANALYZE TABLE derived_atspm;

ALTER TABLE derived_atspm DROP INDEX idx_5min_export;
ALTER TABLE derived_atspm DROP INDEX idx_15min_export;
ALTER TABLE derived_atspm DROP INDEX idx_60min_export;
CREATE INDEX idx_lane_metrics_optimized ON derived_atspm (
    signalid,
    date,
    peak_period,
    direction,
    lane_grouping,
    time_60min  -- Assuming this is the peak_hour source
);

ANALYZE TABLE derived_atspm;

TRUNCATE TABLE lane_metrics;

CALL process_lane_metrics('2024-04-01', '2024-04-07', 1);
CALL process_lane_metrics('2024-04-08', '2024-04-14', 1);
CALL process_lane_metrics('2024-04-15', '2024-04-22', 1);
CALL process_lane_metrics('2024-04-23', '2024-04-30', 1);
CALL process_lane_metrics('2024-05-15', '2024-05-15', 1);
CALL process_lane_metrics('2024-06-01', '2024-06-07', 1);
CALL process_lane_metrics('2024-06-08', '2024-06-14', 1);
CALL process_lane_metrics('2024-06-15', '2024-06-22', 1);
CALL process_lane_metrics('2024-06-23', '2024-06-30', 1);

In [ ]:
## Create tables

In [ ]:
## initial set up (one time)
CREATE TABLE IF NOT EXISTS derived_atspm (
    id INT AUTO_INCREMENT PRIMARY KEY, -- new surrogate key
    signalid INT,
    date DATE,
    interval_no_5m INT,
    time_5min TIME,
    interval_no_15m INT,
    time_15min TIME,
    interval_no_60m INT,
    time_60min TIME,
    peak_period VARCHAR(20),
    direction VARCHAR(20),
    lane_grouping VARCHAR(20),
    eventparam INT,
    volume INT,
    INDEX idx_15min (signalid, date, interval_no_15m, direction, lane_grouping),
    INDEX idx_60min_peak (signalid, date, interval_no_60m, peak_period, direction, lane_grouping),
    INDEX idx_lane_grouping (signalid, date, lane_grouping, direction),
    INDEX idx_time_5min (time_5min),
    INDEX idx_time_15min (time_15min),
    INDEX idx_time_60min (time_60min),
    INDEX idx_peak_period (signalid, date, peak_period, direction, lane_grouping, interval_no_60m)
) ENGINE=InnoDB;

In [ ]:
## initial one-time set-up
CREATE TABLE IF NOT EXISTS aggr_5min (
    signalid INT,
    date DATE,
    time_5min TIME,
    direction VARCHAR(20),
    lane_grouping VARCHAR(20),
    volume INT,
    INDEX idx_export_order (date, signalid)
);

CREATE TABLE IF NOT EXISTS aggr_15min (
    signalid INT,
    date DATE,
    time_15min TIME,
    direction VARCHAR(20),
    lane_grouping VARCHAR(20),
    volume INT,
    INDEX idx_export_order (date, signalid)
);

CREATE TABLE IF NOT EXISTS aggr_60min (
    signalid INT,
    date DATE,
    time_60min TIME,
    direction VARCHAR(20),
    lane_grouping VARCHAR(20),
    volume INT,
    INDEX idx_export_order (date, signalid)
) ;

In [ ]:
## (2) stored procedure to process in date ranges

In [ ]:
## if testing, not batching
TRUNCATE derived_atspm;

In [ ]:
## call with start date and end date
CALL process_derived('2024-04-05', '2024-04-05',1);

In [ ]:
DELIMITER //

CREATE PROCEDURE process_derived(
    IN in_start_date DATE,
    IN in_end_date DATE,
    IN batch_size INT) -- Parameter with default
BEGIN
    DECLARE current_start DATE;
    DECLARE retry_count INT DEFAULT 0;
    DECLARE max_retries INT DEFAULT 3;
    DECLARE batch_success BOOLEAN;
    DECLARE error_message TEXT;

    CREATE TABLE IF NOT EXISTS atspm_process_log (
        log_id INT AUTO_INCREMENT PRIMARY KEY,
        batch_start DATE,
        batch_end DATE,
        status VARCHAR(20),
        rows_processed INT,
        error_message TEXT,
        processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );


    SET SESSION tmp_table_size = 1024 * 1024 * 512;
    SET SESSION max_heap_table_size = 1024 * 1024 * 512;
    SET SESSION net_read_timeout = 3600;
    SET SESSION net_write_timeout = 3600;

    SET current_start = in_start_date;

    my_loop: WHILE current_start <= in_end_date DO
        IF (current_start BETWEEN '2024-05-01' AND '2024-05-14') OR 
           (current_start BETWEEN '2024-05-16' AND '2024-05-31') THEN
            SET current_start = DATE_ADD(current_start, INTERVAL 1 DAY);
            ITERATE my_loop;
        END IF;

        IF current_start = '2024-05-15' THEN
            SET @batch_end = DATE_ADD(current_start, INTERVAL 1 DAY);
        ELSE
            SET @batch_end = DATE_ADD(current_start, INTERVAL batch_size DAY);
            IF MONTH(@batch_end) != MONTH(current_start) THEN
                SET @batch_end = LAST_DAY(current_start);
                SET @batch_end = DATE_ADD(@batch_end, INTERVAL 1 DAY);
            END IF;
        END IF;

        SET retry_count = 0;
        SET batch_success = FALSE;

        batch_loop: WHILE retry_count < max_retries AND batch_success = FALSE DO
            BEGIN
                DECLARE CONTINUE HANDLER FOR SQLEXCEPTION 
                BEGIN
                    GET DIAGNOSTICS CONDITION 1 error_message = MESSAGE_TEXT;
                    INSERT INTO atspm_process_log (batch_start, batch_end, status, error_message)
                    VALUES (current_start, DATE_SUB(@batch_end, INTERVAL 1 DAY), 'failed', error_message);
                    SET retry_count = retry_count + 1;
                    IF retry_count < max_retries THEN
                        DO SLEEP(30 * retry_count);
                    END IF;
                END;

                INSERT INTO derived_atspm (
                    signalid, date, interval_no_5m, time_5min,
                    interval_no_15m, time_15min, interval_no_60m, time_60min,
                    peak_period, direction, lane_grouping, eventparam, volume
                )
                SELECT 
                    signalid,
                    date,
                    FLOOR(timestamp / 300000) AS interval_no_5m,
                    SEC_TO_TIME(FLOOR(timestamp / 300000) * 300) AS time_5min,
                    FLOOR(timestamp / 900000) AS interval_no_15m,
                    SEC_TO_TIME(FLOOR(timestamp / 900000) * 900) AS time_15min,
                    FLOOR(timestamp / 3600000) AS interval_no_60m,
                    SEC_TO_TIME(FLOOR(timestamp / 3600000) * 3600) AS time_60min,
                    CASE
                        WHEN SEC_TO_TIME(FLOOR(timestamp / 1000)) BETWEEN TIME('06:30:00') AND TIME('08:15:00') THEN 'AM Peak'
                        WHEN SEC_TO_TIME(FLOOR(timestamp / 1000)) BETWEEN TIME('11:30:00') AND TIME('13:15:00') THEN 'MD Peak'
                        WHEN SEC_TO_TIME(FLOOR(timestamp / 1000)) BETWEEN TIME('16:30:00') AND TIME('18:15:00') THEN 'PM Peak'
                        ELSE 'Non-Peak'
                    END AS peak_period,
                    CASE 
                        WHEN eventparam IN (58, 59, 60, 61, 62, 64) THEN 'Eastbound'
                        WHEN eventparam IN (42, 43, 44, 45, 46, 48) THEN 'Westbound'
                        WHEN eventparam IN (50, 51, 52, 53, 54, 56) THEN 'Northbound'
                        WHEN eventparam IN (34, 35, 36, 37, 38, 40) THEN 'Southbound'
                        ELSE 'Other'
                    END AS direction,
                    CASE 
                        WHEN eventparam IN (58, 59) THEN 'EB Left'
                        WHEN eventparam IN (60, 61, 62) THEN 'EB Thru'
                        WHEN eventparam IN (64) THEN 'EB Right'
                        WHEN eventparam IN (42, 43) THEN 'WB Left'
                        WHEN eventparam IN (44, 45, 46) THEN 'WB Thru'
                        WHEN eventparam IN (48) THEN 'WB Right'
                        WHEN eventparam IN (50, 51) THEN 'NB Left'
                        WHEN eventparam IN (52, 53, 54) THEN 'NB Thru'
                        WHEN eventparam IN (56) THEN 'NB Right'
                        WHEN eventparam IN (34, 35) THEN 'SB Left'
                        WHEN eventparam IN (36, 37, 38) THEN 'SB Thru'
                        WHEN eventparam IN (40) THEN 'SB Right'
                        ELSE 'Other'
                    END AS lane_grouping,
                    eventparam,
                    1 AS volume
                FROM 
                    raw_atspm
                WHERE 
                    date >= current_start AND date < @batch_end;

                SET batch_success = TRUE;
            END;
        END WHILE;

        DO SLEEP(10);
        SET current_start = @batch_end;
    END WHILE;

    SELECT 
        COUNT(CASE WHEN status = 'success' THEN 1 END) AS successful_batches,
        COUNT(CASE WHEN status = 'failed' THEN 1 END) AS failed_batches,
        SUM(rows_processed) AS total_rows_processed
    FROM atspm_process_log;

    SELECT 'Processing completed with detailed results in atspm_process_log' AS result;
END //

DELIMITER ;


In [ ]:
## for aggregations

In [ ]:
## truncate if testing, not batching 
    TRUNCATE TABLE aggr_5min;
    TRUNCATE TABLE aggr_15min;
    TRUNCATE TABLE aggr_60min;

In [ ]:
CALL process_time_aggr('2024-04-05', '2024-04-05',1);

In [ ]:
DELIMITER //

CREATE PROCEDURE process_time_aggr(IN in_start_date DATE, IN in_end_date DATE, IN batch_size INT)
BEGIN
    DECLARE current_start DATE;
    DECLARE retry_count INT DEFAULT 0;
    DECLARE max_retries INT DEFAULT 3;
    DECLARE batch_success BOOLEAN;
    DECLARE error_message TEXT;

    CREATE TABLE IF NOT EXISTS atspm_process_log (
        log_id INT AUTO_INCREMENT PRIMARY KEY,
        batch_start DATE,
        batch_end DATE,
        status VARCHAR(20),
        rows_processed INT,
        error_message TEXT,
        processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    SET SESSION tmp_table_size = 1024 * 1024 * 512;
    SET SESSION max_heap_table_size = 1024 * 1024 * 512;
    SET SESSION net_read_timeout = 3600;
    SET SESSION net_write_timeout = 3600;

    SET current_start = in_start_date;

my_loop: WHILE current_start <= in_end_date DO
    IF (current_start BETWEEN '2024-05-01' AND '2024-05-14') OR 
       (current_start BETWEEN '2024-05-16' AND '2024-05-31') THEN
        SET current_start = DATE_ADD(current_start, INTERVAL 1 DAY);
        ITERATE my_loop;
    END IF;

    IF current_start = '2024-05-15' THEN
        SET @batch_end = DATE_ADD(current_start, INTERVAL 1 DAY);
    ELSE
        SET @batch_end = DATE_ADD(current_start, INTERVAL batch_size DAY);
        IF MONTH(@batch_end) != MONTH(current_start) THEN
            SET @batch_end = LAST_DAY(current_start);
            SET @batch_end = DATE_ADD(@batch_end, INTERVAL 1 DAY);
        END IF;
    END IF;

    SET retry_count = 0;
    SET batch_success = FALSE;

    batch_loop: WHILE retry_count < max_retries AND batch_success = FALSE DO
        BEGIN
            DECLARE CONTINUE HANDLER FOR SQLEXCEPTION 
            BEGIN
                GET DIAGNOSTICS CONDITION 1 error_message = MESSAGE_TEXT;
                INSERT INTO atspm_process_log (batch_start, batch_end, status, error_message)
                VALUES (current_start, DATE_SUB(@batch_end, INTERVAL 1 DAY), 'failed', error_message);
                SET retry_count = retry_count + 1;
                IF retry_count < max_retries THEN
                    DO SLEEP(30 * retry_count);
                END IF;
            END;

            -- 5 MIN AGGREGATION
            INSERT INTO aggr_5min (
                signalid, date, time_5min, direction, lane_grouping, volume
            )
            SELECT 
                signalid,
                date,
                time_5min,
                direction,
                lane_grouping,
                SUM(volume)
            FROM derived_atspm
            WHERE date >= current_start AND date < @batch_end
            GROUP BY signalid, date, time_5min, direction, lane_grouping;

            -- 15 MIN AGGREGATION
            INSERT INTO aggr_15min (
                signalid, date, time_15min, direction, lane_grouping, volume
            )
            SELECT 
                signalid,
                date,
                time_15min,
                direction,
                lane_grouping,
                SUM(volume)
            FROM derived_atspm
            WHERE date >= current_start AND date < @batch_end
            GROUP BY signalid, date, time_15min, direction, lane_grouping;

            -- 60 MIN AGGREGATION
            INSERT INTO aggr_60min (
                signalid, date, time_60min, direction, lane_grouping, volume
            )
            SELECT 
                signalid,
                date,
                time_60min,
                direction,
                lane_grouping,
                SUM(volume)
            FROM derived_atspm
            WHERE date >= current_start AND date < @batch_end
            GROUP BY signalid, date, time_60min, direction, lane_grouping;

            SET batch_success = TRUE;

            INSERT INTO atspm_process_log (batch_start, batch_end, status, rows_processed)
            VALUES (current_start, DATE_SUB(@batch_end, INTERVAL 1 DAY), 'success', ROW_COUNT());

        END;
    END WHILE;

    DO SLEEP(10);
    SET current_start = @batch_end;
END WHILE;


SELECT 
    COUNT(CASE WHEN status = 'success' THEN 1 END) AS successful_batches,
    COUNT(CASE WHEN status = 'failed' THEN 1 END) AS failed_batches,
    SUM(rows_processed) AS total_rows_processed
FROM atspm_process_log;

SELECT 'Processing completed with detailed results in atspm_process_log' AS result;
END //

DELIMITER ;


In [ ]:
## (3) lane metrics procedure

In [ ]:
## initial set-up (one-time)
CREATE TABLE IF NOT EXISTS lane_metrics (
    id INT AUTO_INCREMENT PRIMARY KEY,
    signalid INT NOT NULL,
    date DATE NOT NULL,
    peak_period VARCHAR(20) NOT NULL,
    direction VARCHAR(10) NOT NULL,
    lane_grouping VARCHAR(100) NOT NULL,
    
    total INT,
    peak_hour TIME,
    peak_hour_volume INT,
    peak_15min TIME,
    peak_15min_volume INT,
    peak_hour_factor DECIMAL(5, 3),
    approach_percent DECIMAL(6, 4),
    
    number_lanes INT,
    vg INT,
    vg1 INT,
    fLU DECIMAL(6, 4)
);


In [ ]:
## clear table if testing, not batching
TRUNCATE lane_metrics;

In [ ]:
CALL process_lane_metrics('2024-05-15', '2024-05-20');

In [ ]:
DELIMITER //
CREATE PROCEDURE process_lane_metrics(IN start_date DATE, IN end_date DATE, IN batch_size INT)
BEGIN
    DECLARE current_start DATE;
    DECLARE retry_count INT DEFAULT 0;
    DECLARE max_retries INT DEFAULT 3;
    DECLARE batch_success BOOLEAN;
    DECLARE error_message TEXT;

    CREATE TABLE IF NOT EXISTS atspm_process_log (
        log_id INT AUTO_INCREMENT PRIMARY KEY,
        batch_start DATE,
        batch_end DATE,
        status VARCHAR(20),
        rows_processed INT,
        error_message TEXT,
        processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    SET SESSION tmp_table_size = 1024 * 1024 * 512;
    SET SESSION max_heap_table_size = 1024 * 1024 * 512;
    SET SESSION net_read_timeout = 3600;
    SET SESSION net_write_timeout = 3600;

    SET current_start = start_date;

my_loop: WHILE current_start <= end_date DO
    IF (current_start BETWEEN '2024-05-01' AND '2024-05-14') OR 
       (current_start BETWEEN '2024-05-16' AND '2024-05-31') THEN
        SET current_start = DATE_ADD(current_start, INTERVAL 1 DAY);
        ITERATE my_loop;
    END IF;

    IF current_start = '2024-05-15' THEN
        SET @batch_end = DATE_ADD(current_start, INTERVAL 1 DAY);
    ELSE
        SET @batch_end = DATE_ADD(current_start, INTERVAL batch_size DAY);
        IF MONTH(@batch_end) != MONTH(current_start) THEN
            SET @batch_end = LAST_DAY(current_start);
            SET @batch_end = DATE_ADD(@batch_end, INTERVAL 1 DAY);
        END IF;
    END IF;

    SET retry_count = 0;
    SET batch_success = FALSE;

    batch_loop: WHILE retry_count < max_retries AND batch_success = FALSE DO
        BEGIN
            DECLARE CONTINUE HANDLER FOR SQLEXCEPTION 
            BEGIN
                GET DIAGNOSTICS CONDITION 1 error_message = MESSAGE_TEXT;
                INSERT INTO atspm_process_log (batch_start, batch_end, status, error_message)
                VALUES (current_start, DATE_SUB(@batch_end, INTERVAL 1 DAY), 'failed', error_message);
                SET retry_count = retry_count + 1;
                IF retry_count < max_retries THEN
                    DO SLEEP(30 * retry_count);
                END IF;
            END;

            -- lane_metrics
            INSERT INTO lane_metrics (
                signalid, date, peak_period, direction, lane_grouping, total, peak_hour, peak_hour_volume,
                peak_15min, peak_15min_volume, peak_hour_factor, approach_percent, number_lanes, vg, vg1, fLU
            )
            WITH peak_hour_data AS (
                SELECT DISTINCT
                    signalid, date, peak_period, direction, lane_grouping,
                    FIRST_VALUE(time_60min) OVER (PARTITION BY signalid, date, peak_period, direction, lane_grouping ORDER BY SUM(volume) DESC) AS peak_hour,
                    FIRST_VALUE(SUM(volume)) OVER (PARTITION BY signalid, date, peak_period, direction, lane_grouping ORDER BY SUM(volume) DESC) AS peak_hour_volume
                FROM derived_atspm
                WHERE peak_period IN ('AM Peak', 'MD Peak', 'PM Peak')
                GROUP BY signalid, date, peak_period, time_60min, direction, lane_grouping
            ),
            peak_15min_data AS (
                SELECT
                    p.signalid, p.date, p.peak_period, p.direction, p.lane_grouping, p.peak_hour, p.peak_hour_volume,
                    time_15min,
                    SUM(volume) AS total_15min_volume,
                    FIRST_VALUE(time_15min) OVER (
                        PARTITION BY p.signalid, p.date, p.peak_period, p.direction, p.lane_grouping, p.peak_hour
                        ORDER BY SUM(d.volume) DESC
                    ) AS peak_15min,
                    FIRST_VALUE(SUM(d.volume)) OVER (
                        PARTITION BY p.signalid, p.date, p.peak_period, p.direction, p.lane_grouping, p.peak_hour
                        ORDER BY SUM(d.volume) DESC
                    ) AS peak_15min_volume
                FROM derived_atspm d
                JOIN peak_hour_data p ON d.signalid = p.signalid AND d.date = p.date AND d.peak_period = p.peak_period
                    AND d.direction = p.direction AND d.lane_grouping = p.lane_grouping AND d.time_60min = p.peak_hour
                GROUP BY p.signalid, p.date, p.peak_period, p.direction, p.lane_grouping, p.peak_hour, p.peak_hour_volume, time_15min
            ),
            lanes AS (
                SELECT DISTINCT lane_grouping, COUNT(DISTINCT eventparam) AS number_lanes
                FROM derived_atspm
                GROUP BY lane_grouping
            ),
            aggr_eventparam AS (
                SELECT
                    p.signalid, p.date, p.peak_period, p.peak_hour, p.direction, p.lane_grouping, d.eventparam,
                    SUM(d.volume) AS lane_total
                FROM derived_atspm d
                JOIN peak_hour_data p ON d.signalid = p.signalid AND d.date = p.date AND d.peak_period = p.peak_period
                    AND d.direction = p.direction AND d.lane_grouping = p.lane_grouping AND d.time_60min = p.peak_hour
                GROUP BY signalid, date, peak_period, peak_hour, direction, lane_grouping, eventparam
            ),
            vg1 AS (
                SELECT
                    p.signalid, p.date, p.peak_period, p.direction, p.lane_grouping,
                    SUM(lane_total) OVER (PARTITION BY a.signalid, a.date, a.peak_period, a.direction, a.lane_grouping) AS total,
                    p.peak_hour, p.peak_hour_volume,
                    FIRST_VALUE(a.lane_total) OVER (PARTITION BY signalid, date, peak_period, direction, lane_grouping ORDER BY a.lane_total DESC) AS vg1,
                    (SUM(a.lane_total) OVER (PARTITION BY a.signalid, a.date, a.peak_period, a.direction, a.lane_grouping) /
                     SUM(a.lane_total) OVER (PARTITION BY a.signalid, a.date, a.peak_period, a.direction)) AS approach_percent
                FROM aggr_eventparam a
                JOIN peak_hour_data p ON a.signalid = p.signalid AND a.date = p.date AND a.peak_period = p.peak_period
                    AND a.direction = p.direction AND a.lane_grouping = p.lane_grouping
            )
            SELECT DISTINCT
                v.signalid, v.date, v.peak_period, v.direction, v.lane_grouping, v.total, v.peak_hour, v.peak_hour_volume,
                p.peak_15min, p.peak_15min_volume,
                (v.peak_hour_volume / (4 * p.peak_15min_volume)) AS peak_hour_factor,
                v.approach_percent, l.number_lanes, v.peak_hour_volume AS vg, v.vg1,
                v.peak_hour_volume / (v.vg1 * l.number_lanes) AS fLU
            FROM vg1 AS v
            JOIN lanes AS l ON v.lane_grouping = l.lane_grouping
            JOIN peak_15min_data p ON v.signalid = p.signalid AND v.date = p.date AND v.peak_period = p.peak_period
                AND v.direction = p.direction AND v.lane_grouping = p.lane_grouping AND v.peak_hour = p.peak_hour
            GROUP BY v.signalid, v.date, v.peak_period, v.direction, v.lane_grouping, v.peak_hour, l.number_lanes,
                     v.peak_hour_volume, p.peak_15min, p.peak_15min_volume, v.vg1, v.approach_percent, v.total
            ORDER BY v.signalid, v.date, v.peak_period, v.direction, v.lane_grouping, v.peak_hour, l.number_lanes;

            SET batch_success = TRUE;

            INSERT INTO atspm_process_log (batch_start, batch_end, status, rows_processed)
            VALUES (current_start, DATE_SUB(@batch_end, INTERVAL 1 DAY), 'success', ROW_COUNT());

        END;
    END WHILE;

    DO SLEEP(10);
    SET current_start = @batch_end;
END WHILE;

OPTIMIZE TABLE aggr_5min, aggr_15min, aggr_60min;

SELECT 
    COUNT(CASE WHEN status = 'success' THEN 1 END) AS successful_batches,
    COUNT(CASE WHEN status = 'failed' THEN 1 END) AS failed_batches,
    SUM(rows_processed) AS total_rows_processed
FROM atspm_process_log;

SELECT 'Processing completed with detailed results in atspm_process_log' AS result;
END //
DELIMITER ;

In [ ]:
##(3) execute batch processing

-- Call the stored procedure
CALL ProcessAtspmData();

-- Or process a specific date range manually
INSERT INTO derived_atspm
SELECT ...  -- same query as above
WHERE date BETWEEN '2024-01-01' AND '2024-01-31';

In [ ]:
ALTER TABLE derived_atspm DISABLE KEYS;
-- Run your inserts
ALTER TABLE derived_atspm ENABLE KEYS;

In [ ]:
## memory modifications
SET GLOBAL tmp_table_size = 1024*1024*256; -- 256MB
SET GLOBAL max_heap_table_size = 1024*1024*256;
-- In MySQL: These can be changed without restart
SET GLOBAL innodb_buffer_pool_size = 268435456;  -- 256MB (temporary)
SET GLOBAL innodb_lock_wait_timeout = 120;
SET GLOBAL innodb_io_capacity = 2000;